# 🏠 SQL Analysis — Real Estate Sales Optimisation
**Author:** Kresio Azevedo Fernando  
**Portfolio:** [kresio-azevedo-fernando.github.io](https://kresio-azevedo-fernando.github.io)

---

> ⚠️ **Note on data:** This notebook uses the **real anonymised dataset** of 500 clients from a real estate agency. All KPIs, conversion rates and salary figures match exactly those visible in the live Power BI dashboard. Client identifiers are anonymised.

---

## Business Problem
Analysis of **500 clients** from a real estate agency.  
Only **30% (148 clients) purchased** — 70% (352) did not convert.  
Generic campaigns ignoring age and salary propensity profiles.

**This notebook answers:**
1. **What happened?** — Who bought and who did not?
2. **Why?** — What profile drives conversion?
3. **What to do?** — Simplex + Dijkstra for +€1.4M impact

---

In [ ]:
!pip install pandas matplotlib seaborn -q

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': '#09090f', 'axes.facecolor': '#0f0f1a',
    'axes.labelcolor': '#e8e8f0', 'xtick.color': '#9494a8',
    'ytick.color': '#9494a8', 'text.color': '#e8e8f0',
    'axes.titlecolor': '#e8e8f0', 'axes.edgecolor': '#1a1a28',
    'grid.color': '#1a1a28', 'axes.grid': True,
})
ACCENT = '#bb9476'
BLUE   = '#6eb5ff'
GREEN  = '#1D9E75'
RED    = '#ff6b6b'
print('✓ Setup complete')

In [ ]:
# ── REAL DATASET — 500 clients from dashboard ─────────────────
conn = sqlite3.connect(':memory:')

# Real client data from dashboard
real_data = [
    [1,1,38,90694.82,1],[2,0,49,65455.36,1],[3,1,40,72723.70,0],
    [4,1,50,82079.56,1],[5,1,20,57367.59,1],[6,0,35,22388.22,1],
    [7,1,42,60654.06,1],[8,1,59,64881.50,0],[9,1,48,78362.36,0],
    [10,0,20,51691.06,0],[11,1,57,25844.33,1],[12,1,41,39120.95,0],
    [13,1,49,45862.00,1],[14,1,39,69902.97,1],[15,0,40,63516.16,0],
    [16,1,19,45268.22,1],[17,0,44,98853.80,0],[18,0,59,66490.86,0],
    [19,0,19,35164.28,1],[20,1,43,23651.51,0],[21,0,34,27993.03,1],
    [22,1,57,35906.41,1],[23,0,50,28657.92,0],[24,0,26,30858.20,0],
    [25,0,56,39233.09,0],[26,0,46,29736.76,0],[27,0,59,91225.06,1],
    [28,0,43,21819.87,0],[29,0,52,59583.47,0],[30,0,42,49883.73,0],
    [31,1,41,98502.18,0],[32,1,30,24523.31,1],[33,0,24,48817.73,0],
    [34,0,53,97404.99,1],[35,0,37,88568.11,0],[36,1,18,84451.13,1],
    [37,0,25,36921.74,0],[38,1,33,29525.44,0],[39,1,31,71834.67,1],
    [40,1,29,93996.96,0],[41,1,40,62324.85,1],[42,1,32,63587.08,0],
    [43,0,45,38798.22,0],[44,0,51,80406.90,0],[45,0,19,30898.72,1],
    [46,0,49,42512.74,0],[47,0,40,51162.10,0],[48,1,39,58146.88,0],
    [49,0,42,35604.83,0],[50,0,39,24761.13,0],[51,1,39,66902.70,1],
    [52,0,59,39533.60,1],[53,1,23,64405.25,1],[54,0,32,28120.83,1],
    [55,1,54,55896.91,0],[56,0,50,60270.10,1],[57,0,25,19405.00,0],
    [58,1,22,43611.36,0],[59,1,56,26425.25,0],[60,1,21,20386.87,0],
    [61,1,23,99146.62,1],[62,1,49,42400.08,0],[63,1,47,83839.33,0],
    [64,1,52,36644.46,0],[65,1,57,72927.73,1],[66,0,33,79619.37,0],
    [67,0,30,65629.29,0],[68,1,59,55083.98,0],[69,0,47,50006.48,1],
    [70,0,36,44653.80,0],[71,0,34,94009.98,0],[72,0,36,85602.65,0],
    [73,1,45,97027.29,0],[74,0,43,25565.26,0],[75,1,54,77123.74,0],
    [76,0,43,94758.94,1],[77,0,40,30404.81,0],[78,0,26,20652.18,0],
    [79,1,29,77995.26,1],[80,0,18,63830.21,0],[81,1,18,86555.45,0],
    [82,0,51,26880.65,0],[83,1,49,82597.72,0],[84,0,42,32138.32,1],
    [85,1,57,28910.76,0],[86,1,18,28962.59,1],[87,0,33,84238.85,0],
    [88,1,56,71541.76,0],[89,0,22,59460.56,0],[90,0,39,45500.59,0],
    [91,0,46,89562.05,0],[92,0,20,48357.83,1],[93,0,29,84410.95,0],
    [94,0,43,52326.47,1],[95,0,33,47040.28,0],[96,0,54,54327.78,1],
    [97,0,39,40617.12,1],[98,0,46,78546.80,1],[99,0,31,57731.23,1],
    [100,1,45,34738.08,1],[101,1,22,91463.84,0],[102,0,47,47630.75,0],
    [103,0,22,61201.99,0],[104,0,29,92050.13,1],[105,0,33,68060.23,0],
    [106,0,43,24936.33,1],[107,0,43,94885.73,0],[108,0,38,68355.18,0],
    [109,0,56,43466.98,0],[110,1,53,26838.13,0],[111,0,50,82492.14,0],
    [112,1,47,67706.18,0],[113,0,54,60344.19,0],[114,0,40,90980.87,0],
    [115,1,27,82030.76,0],[116,0,22,27892.36,0],[117,1,53,41496.38,1],
    [118,0,51,36121.58,0],[119,0,48,78235.43,0],[120,1,27,17850.26,0],
    [121,0,36,63440.62,0],[122,1,49,79808.99,0],[123,0,18,89525.08,0],
    [124,1,22,44076.95,0],[125,1,21,84806.87,0],[126,0,33,24403.70,0],
    [127,0,41,86948.44,1],[128,1,33,25836.54,1],[129,0,19,48769.42,0],
    [130,0,45,82770.11,0],[131,0,49,27742.98,0],[132,1,44,34486.37,0],
    [133,1,37,76391.47,1],[134,1,41,76203.11,0],[135,1,29,69497.55,0],
    [136,1,52,73985.62,0],[137,1,50,61131.58,1],[138,1,50,36402.92,0],
    [139,1,54,44384.16,0],[140,1,29,30435.81,0],[141,0,20,92218.30,0],
    [142,1,18,64588.30,0],[143,0,50,49072.37,0],[144,0,57,54270.49,0],
    [145,0,27,95519.08,0],[146,1,46,28034.87,0],[147,1,30,64829.54,0],
    [148,1,29,58000.54,0],[149,1,48,66973.61,0],[150,0,19,16539.37,1],
    [151,1,52,89130.53,0],[152,1,40,94230.05,0],[153,1,34,63036.32,0],
    [154,1,43,74215.32,0],[155,1,25,93412.45,1],[156,0,46,75115.28,0],
    [157,1,43,27965.82,1],[158,0,27,63984.51,0],[159,1,43,66570.78,0],
    [160,0,51,51051.11,1],[161,1,58,77597.76,1],[162,1,24,94421.20,1],
    [163,0,21,93673.32,1],[164,0,28,53321.35,0],[165,0,46,24625.23,1],
    [166,1,53,98711.50,0],[167,0,42,86306.34,1],[168,1,38,25596.33,0],
    [169,1,53,93271.56,1],[170,0,27,88941.19,0],[171,0,54,59101.23,0],
    [172,1,26,65258.41,0],[173,1,41,48915.23,1],[174,0,52,19654.74,0],
    [175,0,52,43491.77,0],[176,0,53,83242.54,0],[177,1,35,15393.72,0],
    [178,1,56,43347.43,1],[179,1,49,48844.34,0],[180,1,41,60678.63,1],
    [181,1,40,93187.73,0],[182,1,49,44439.41,0],[183,0,54,44491.02,1],
    [184,1,29,77687.61,0],[185,1,30,53438.52,0],[186,1,40,34091.41,0],
    [187,0,42,53457.36,1],[188,1,52,26972.85,0],[189,1,58,29992.89,1],
    [190,0,47,57361.26,1],[191,1,34,50608.66,0],[192,1,37,92761.90,0],
    [193,1,42,45803.48,0],[194,1,39,64350.01,0],[195,1,30,68742.46,0],
    [196,0,36,16113.03,0],[197,0,53,71400.68,0],[198,0,29,30133.06,1],
    [199,1,58,96690.98,1],[200,1,36,27636.33,1],[201,1,29,50243.05,0],
    [202,0,26,22254.72,0],[203,1,24,99734.31,0],[204,1,45,57686.58,1],
    [205,0,31,65607.73,0],[206,1,48,20701.50,1],[207,0,36,78746.64,0],
    [208,0,33,32841.98,0],[209,0,22,91334.61,1],[210,1,52,32436.87,0],
    [211,1,29,31208.46,1],[212,1,42,18106.72,1],[213,1,38,55125.69,0],
    [214,1,53,63011.50,0],[215,1,40,20585.23,0],[216,1,33,80919.85,1],
    [217,0,56,53529.55,0],[218,1,59,59573.17,0],[219,0,56,52464.83,0],
    [220,1,31,49064.86,1],[221,1,48,62569.43,0],[222,1,22,28195.42,1],
    [223,0,52,30463.89,1],[224,0,40,88251.78,0],[225,0,46,95419.81,0],
    [226,0,28,46731.29,1],[227,1,35,38013.30,0],[228,0,29,69739.96,1],
    [229,1,26,49742.40,0],[230,1,27,17157.84,0],[231,0,34,28272.97,1],
    [232,0,55,75857.64,0],[233,0,24,71008.54,0],[234,0,30,17303.16,0],
    [235,0,57,33867.63,0],[236,0,59,34641.36,0],[237,0,26,72110.88,0],
    [238,0,44,16675.40,1],[239,1,19,23849.23,0],[240,0,22,82992.87,0],
    [241,0,46,30176.30,0],[242,1,54,70483.42,1],[243,0,55,35245.54,0],
    [244,1,36,23452.52,1],[245,1,25,35669.64,0],[246,0,18,76392.69,0],
    [247,1,39,87734.20,0],[248,1,34,85568.69,0],[249,1,24,48760.60,0],
    [250,1,42,71787.24,0],[251,0,21,32423.67,1],[252,1,53,39917.56,0],
    [253,0,23,91188.54,0],[254,1,48,16105.16,0],[255,1,36,22268.23,1],
    [256,1,56,32670.33,0],[257,1,44,17255.24,0],[258,0,27,30422.01,0],
    [259,0,43,64558.53,1],[260,1,36,50821.09,0],[261,1,56,90877.10,0],
    [262,0,20,84482.70,1],[263,1,30,44054.47,1],[264,1,45,37050.99,0],
    [265,1,37,47273.85,1],[266,0,45,65175.07,0],[267,0,25,37785.41,0],
    [268,0,58,68052.66,0],[269,1,56,49799.99,0],[270,1,18,61924.01,0],
    [271,0,20,52070.75,1],[272,0,30,40029.59,0],[273,0,45,95618.53,0],
    [274,0,42,79906.49,0],[275,1,50,26909.62,1],[276,0,55,88819.78,0],
    [277,1,23,56431.65,0],[278,0,49,91036.94,0],[279,1,38,82987.70,0],
    [280,0,33,51143.15,1],[281,0,38,16909.89,1],[282,0,28,37837.58,0],
    [283,0,54,61038.91,1],[284,1,53,68845.65,0],[285,0,52,36920.45,1],
    [286,1,36,26845.27,0],[287,1,37,85969.07,0],[288,1,35,98674.19,0],
    [289,1,58,59683.67,0],[290,0,31,29592.74,0],[291,1,32,38146.12,0],
    [292,1,48,16563.21,0],[293,1,18,92715.40,1],[294,0,20,25008.84,0],
    [295,0,33,64003.90,0],[296,0,40,38294.69,1],[297,0,28,62105.13,0],
    [298,1,29,70370.73,0],[299,1,27,85528.05,0],[300,0,49,32545.81,1],
    [301,1,33,15934.65,0],[302,1,25,26635.28,0],[303,1,55,91501.58,1],
    [304,0,29,89280.66,1],[305,0,41,65780.11,0],[306,1,45,66043.93,0],
    [307,0,25,71528.12,0],[308,0,45,29906.56,1],[309,0,53,92725.02,0],
    [310,0,43,50595.49,0],[311,0,25,47566.77,1],[312,1,45,59108.00,0],
    [313,0,45,18992.11,0],[314,1,54,29134.09,0],[315,1,58,77732.86,0],
    [316,0,53,22037.89,0],[317,1,44,66267.93,0],[318,1,34,35854.67,0],
    [319,1,26,48090.13,0],[320,0,50,39538.97,1],[321,0,37,45232.18,0],
    [322,1,30,76118.90,1],[323,0,45,40255.35,0],[324,0,46,63144.39,1],
    [325,0,30,55464.28,1],[326,0,52,71412.05,0],[327,1,23,94630.53,1],
    [328,1,35,77268.63,1],[329,0,22,33269.93,0],[330,0,42,17650.57,0],
    [331,1,19,37292.44,0],[332,1,27,65581.62,1],[333,0,47,19371.19,1],
    [334,1,22,57191.13,1],[335,0,50,65731.64,0],[336,0,18,43410.73,1],
    [337,1,35,80527.54,0],[338,1,49,24060.85,0],[339,0,28,21386.71,0],
    [340,0,38,76896.04,0],[341,1,43,57116.76,0],[342,0,42,73514.20,0],
    [343,1,39,51960.32,0],[344,0,44,35944.17,0],[345,1,30,84623.70,0],
    [346,1,50,82950.35,0],[347,1,51,74049.20,0],[348,0,58,38132.34,0],
    [349,0,52,65169.61,0],[350,1,18,45682.78,0],[351,0,38,22784.48,0],
    [352,1,23,92971.65,0],[353,1,45,26629.58,0],[354,0,34,95770.18,0],
    [355,0,22,52910.49,1],[356,1,48,30736.30,0],[357,0,22,61061.58,0],
    [358,0,55,89200.40,0],[359,1,20,77239.12,0],[360,1,40,83557.70,0],
    [361,0,54,70996.59,0],[362,1,54,73843.51,0],[363,1,27,87181.63,0],
    [364,0,27,36221.78,0],[365,1,36,56601.12,1],[366,1,34,33802.80,1],
    [367,0,38,98951.78,1],[368,0,31,95245.04,1],[369,1,26,18351.28,0],
    [370,1,18,74973.89,0],[371,0,30,93646.11,0],[372,0,21,30348.90,0],
    [373,0,18,63275.34,1],[374,0,57,92816.51,0],[375,1,49,17885.41,1],
    [376,0,51,74280.72,0],[377,0,45,40274.67,1],[378,1,48,93573.68,0],
    [379,0,25,97539.95,0],[380,1,56,95262.65,0],[381,0,43,55308.21,0],
    [382,0,51,88273.63,0],[383,0,20,86786.70,0],[384,1,29,42123.54,1],
    [385,1,18,85457.82,0],[386,0,22,18145.65,0],[387,1,47,65682.94,0],
    [388,1,47,34550.75,1],[389,1,34,25248.19,0],[390,1,40,21541.02,0],
    [391,1,32,74184.55,0],[392,1,54,43889.37,0],[393,0,38,76605.18,0],
    [394,0,31,20555.29,0],[395,1,19,41799.68,0],[396,0,28,60856.76,1],
    [397,0,56,82211.47,0],[398,0,55,42093.96,0],[399,0,51,68200.77,0],
    [400,1,55,90308.11,1],[401,0,51,67348.37,0],[402,0,35,34801.56,1],
    [403,1,47,17074.07,0],[404,1,32,88958.40,0],[405,0,44,16807.90,0],
    [406,0,51,89349.64,1],[407,0,55,59959.66,0],[408,0,50,94820.75,1],
    [409,0,41,82896.58,1],[410,0,32,99824.40,0],[411,1,47,44810.50,0],
    [412,0,59,80211.00,0],[413,0,34,49164.13,0],[414,1,22,55789.43,1],
    [415,1,46,68337.96,1],[416,1,21,89262.55,0],[417,1,27,98647.09,0],
    [418,0,34,80303.24,1],[419,0,27,50510.18,0],[420,0,34,50815.35,1],
    [421,0,37,77694.50,0],[422,0,41,35296.06,0],[423,0,22,24390.30,0],
    [424,1,51,45142.88,0],[425,0,23,39415.31,0],[426,1,19,40186.19,0],
    [427,1,30,34856.66,0],[428,0,28,18577.92,1],[429,1,40,16519.28,1],
    [430,0,33,98956.40,0],[431,1,48,51360.72,0],[432,0,28,47667.77,0],
    [433,1,33,72770.02,0],[434,0,25,33551.58,0],[435,0,21,95746.70,0],
    [436,0,57,81839.33,0],[437,0,21,22599.94,0],[438,0,42,50494.37,0],
    [439,1,20,89725.06,0],[440,1,49,95302.22,1],[441,1,20,54729.13,0],
    [442,0,44,67139.97,0],[443,1,46,29197.89,0],[444,0,49,99249.33,0],
    [445,0,36,34692.09,1],[446,1,38,95132.20,0],[447,1,22,70219.97,1],
    [448,0,35,66657.63,0],[449,1,45,58578.52,0],[450,0,59,34606.93,1],
    [451,0,39,30004.88,0],[452,0,38,33741.33,0],[453,0,23,30847.25,0],
    [454,0,18,81264.68,0],[455,1,22,44760.65,1],[456,1,58,19916.63,0],
    [457,0,29,97373.72,0],[458,0,43,90121.80,0],[459,1,51,93858.94,0],
    [460,1,31,99567.16,0],[461,0,43,29781.10,0],[462,1,44,48680.57,0],
    [463,0,26,79450.27,1],[464,1,43,74161.75,0],[465,0,39,28081.15,0],
    [466,1,47,84345.82,1],[467,0,34,34077.45,0],[468,1,43,34024.50,0],
    [469,1,53,60642.83,0],[470,0,18,65399.89,0],[471,1,25,64307.33,0],
    [472,0,52,22776.38,1],[473,1,32,89584.17,0],[474,0,39,37576.00,0],
    [475,1,31,26008.77,1],[476,0,43,90543.59,0],[477,1,45,96230.38,0],
    [478,0,40,88280.85,0],[479,1,31,83808.87,1],[480,0,41,70695.57,0],
    [481,0,19,61822.88,0],[482,0,43,22393.87,0],[483,1,31,49718.52,0],
    [484,0,24,46678.52,0],[485,1,20,37079.07,0],[486,0,40,76490.71,1],
    [487,1,35,57149.44,0],[488,0,55,21888.93,0],[489,1,52,33715.57,1],
    [490,1,32,73076.99,1],[491,0,42,21471.12,0],[492,1,54,87352.59,0],
    [493,1,45,57087.45,0],[494,0,27,55849.86,1],[495,1,56,65354.66,1],
    [496,1,34,85097.88,0],[497,1,56,44563.78,1],[498,0,39,72631.37,0],
    [499,1,43,63087.22,0],[500,0,42,37697.40,0],
]

clients = pd.DataFrame(real_data,
    columns=['client_id','gender','age','salary','purchased'])

clients['age_group'] = pd.cut(
    clients['age'],
    bins=[17,25,30,35,40,45,50,60],
    labels=['18-25','26-30','31-35','36-40','41-45','46-50','51-60']
)
clients['gender_label'] = clients['gender'].map({1:'Male', 0:'Female'})
clients['salary_band'] = pd.cut(
    clients['salary'],
    bins=[0,30000,45000,60000,80000,999999],
    labels=['<30K','30-45K','45-60K','60-80K','80K+']
)

clients.to_sql('clients', conn, index=False, if_exists='replace')

buyers    = clients[clients['purchased']==1]
nonbuyers = clients[clients['purchased']==0]

print(f'✓ Real dataset loaded — {len(clients)} clients')
print(f'  Buyers:     {len(buyers)} ({len(buyers)/len(clients)*100:.0f}%)')
print(f'  Non-buyers: {len(nonbuyers)} ({len(nonbuyers)/len(clients)*100:.0f}%)')
print(f'  Avg buyer salary:     €{buyers["salary"].mean():,.0f}')
print(f'  Avg non-buyer salary: €{nonbuyers["salary"].mean():,.0f}')

---
## Query 1 — DESCRIPTIVE: Conversion overview
**Who bought and who did not?**

In [ ]:
q1 = """
SELECT
    CASE WHEN purchased=1 THEN 'Purchased' ELSE 'Did not purchase' END AS status,
    COUNT(*)                              AS total,
    ROUND(AVG(age),1)                     AS avg_age,
    ROUND(AVG(salary),0)                  AS avg_salary,
    ROUND(COUNT(*)*100.0/500,1)           AS pct_of_total,
    SUM(CASE WHEN gender=1 THEN 1 ELSE 0 END) AS male,
    SUM(CASE WHEN gender=0 THEN 1 ELSE 0 END) AS female
FROM clients
GROUP BY purchased
ORDER BY purchased DESC;
"""
df1 = pd.read_sql(q1, conn)
print('📊 CONVERSION OVERVIEW (Real Data — 500 clients)')
print(df1.to_string(index=False))

b  = df1[df1['status']=='Purchased'].iloc[0]
nb = df1[df1['status']=='Did not purchase'].iloc[0]
print(f'\n🔍 INSIGHT:')
print(f'   Buyer avg salary:     €{int(b["avg_salary"]):,} (dashboard: €55,338)')
print(f'   Non-buyer avg salary: €{int(nb["avg_salary"]):,} (dashboard: €57,854)')
print(f'   Buyers have LOWER salary → accessible properties have more appeal')
print(f'   Gender split: Male {b["male"]+nb["male"]} (51%) · Female {b["female"]+nb["female"]} (49%)')

---
## Query 2 — DIAGNOSTIC: Conversion by age group
**Which age groups convert most?**

In [ ]:
q2 = """
SELECT
    age_group,
    COUNT(*)                                    AS total,
    SUM(purchased)                              AS buyers,
    ROUND(SUM(purchased)*100.0/COUNT(*),1)      AS conv_rate_pct,
    ROUND(AVG(salary),0)                        AS avg_salary
FROM clients
WHERE age_group IS NOT NULL
GROUP BY age_group
ORDER BY conv_rate_pct DESC;
"""
df2 = pd.read_sql(q2, conn)
print('📊 CONVERSION RATE BY AGE GROUP (Real Data)')
print(df2.to_string(index=False))

top = df2.iloc[0]
print(f'\n🔍 INSIGHT: Highest conversion — age group {top["age_group"]} at {top["conv_rate_pct"]}%')
print(f'   → These segments are Simplex allocation Priority 1')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_bar = [ACCENT if r >= 35 else '#2d2d3a' for r in df2['conv_rate_pct']]
axes[0].bar(df2['age_group'], df2['conv_rate_pct'], color=colors_bar, edgecolor='#1a1a28')
axes[0].axhline(df2['conv_rate_pct'].mean(), color=BLUE, linestyle='--',
                linewidth=1.5, label=f'Avg {df2["conv_rate_pct"].mean():.1f}%')
axes[0].set_title('Conversion Rate by Age Group (%)', fontsize=11)
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].legend(fontsize=8)

axes[1].bar(df2['age_group'], df2['buyers'], color=ACCENT, edgecolor='#1a1a28', label='Buyers')
axes[1].bar(df2['age_group'],
            df2['total'] - df2['buyers'],
            bottom=df2['buyers'],
            color='#2d2d3a', edgecolor='#1a1a28', label='Non-buyers')
axes[1].set_title('Total Clients by Age Group', fontsize=11)
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Number of Clients')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('q2_age_conversion.png', dpi=150, bbox_inches='tight', facecolor='#09090f')
plt.show()

---
## Query 3 — DIAGNOSTIC: Salary band analysis
**Does salary predict purchase behaviour?**

In [ ]:
q3 = """
SELECT
    salary_band,
    COUNT(*)                                AS total,
    SUM(purchased)                          AS buyers,
    ROUND(SUM(purchased)*100.0/COUNT(*),1)  AS conv_rate_pct,
    ROUND(AVG(salary),0)                    AS avg_salary
FROM clients
WHERE salary_band IS NOT NULL
GROUP BY salary_band
ORDER BY salary_band;
"""
df3 = pd.read_sql(q3, conn)
print('📊 CONVERSION BY SALARY BAND (Real Data)')
print(df3.to_string(index=False))

print(f'\n🔍 INSIGHT: Avg buyer salary €55,338 < non-buyer €57,854')
print(f'   Lower salary segments are NOT lower propensity')
print(f'   → Campaigns should NOT target only high earners')
print(f'   → Mid-salary 30-60K segments have strong conversion potential')

---
## Query 4 — PRESCRIPTIVE: Priority segments for Simplex
**Which segments to target for maximum ROI on €100K/month marketing budget?**

In [ ]:
q4 = """
SELECT
    age_group,
    gender_label,
    COUNT(*)                                   AS total,
    SUM(purchased)                             AS buyers,
    ROUND(SUM(purchased)*100.0/COUNT(*),1)     AS conv_rate_pct,
    ROUND(AVG(salary),0)                       AS avg_salary,
    CASE
        WHEN SUM(purchased)*100.0/COUNT(*) >= 40
        THEN 'PRIORITY 1 — Max allocation'
        WHEN SUM(purchased)*100.0/COUNT(*) >= 30
        THEN 'PRIORITY 2 — High allocation'
        ELSE 'STANDARD'
    END AS simplex_priority
FROM clients
WHERE age_group IS NOT NULL
GROUP BY age_group, gender_label
HAVING total >= 8
ORDER BY conv_rate_pct DESC
LIMIT 12;
"""
df4 = pd.read_sql(q4, conn)
print('📊 SEGMENT PRIORITY MATRIX — Simplex Input (Real Data)')
print(df4.to_string(index=False))

p1 = df4[df4['simplex_priority'].str.startswith('PRIORITY 1')]
print(f'\n🔍 INSIGHT: {len(p1)} segment(s) at Priority 1 — maximum Simplex budget allocation')
print(f'   Run simplex_model.py for exact optimal budget split → +23% conversions → +€640K/year')

In [ ]:
print('=' * 60)
print(' BUSINESS CONCLUSIONS — SQL ANALYSIS (REAL DATA)')
print('=' * 60)
print(f"""
1. WHAT HAPPENED
   500 clients · 148 buyers (30%) · 352 non-buyers (70%).
   Avg buyer salary €55,338 vs non-buyer €57,854.
   Gender: Male 51% · Female 49% — near equal split.

2. WHERE THE PROBLEM IS
   Generic campaigns ignoring age and salary propensity.
   Agent routes random — excessive travel costs.
   No personalisation by buyer profile.

3. WHY IT HAPPENS
   Budget allocated equally across all segments.
   Highest-propensity age groups underfunded.
   Agents visit clients by registration order.

4. WHAT TO DO (Simplex + Dijkstra)
   Marketing segmentation (Simplex):  +€0.64M (+23% conv.)
   Agent route optimisation (Dijkstra): +€0.22M (−28% dist.)
   Personalised approaches:            +€0.54M
   ──────────────────────────────────────────────────────
   TOTAL: €4.4M → €5.8M · +€1.4M · ROI 320%
   €224K/year saved in agent travel costs (Dijkstra)
""")
print('=' * 60)
print('Live dashboard: https://app.powerbi.com/view?r=eyJrIjoiMTQ5NGVkNGEtM2UwOS00NmIxLTgzYzgtYjg4ZWVmZjBhNmY2IiwidCI6IjY1OWNlMmI4LTA3MTQtNDE5OC04YzM4LWRjOWI2MGFhYmI1NyJ9')
print('Portfolio: kresio-azevedo-fernando.github.io')